# 03 - Data Validation: Evidence for the Modelling Dataset

**Project:** Online Retail II - Customer Churn Prediction
**Role:** Data Engineering & Cleaning

| Task | Where |
|---|---|
| 6. Complete and verify the data validation notebook | This notebook |
| 4. Check missing values, duplicate customers and invalid records | Sections A and C |
| 5. Ensure correct customer IDs and reference dates | Section B |
| 1. Verify the cleaned transaction dataset (re-checked) | Section A |

This notebook **does not reuse the code of notebook 02**. It reloads the saved files and recalculates every customer,
every feature and the churn target with its own code, then compares them row by row.

**Input:** `online_retail_II.csv` and the `Working_Data` files from notebooks 01 and 02

**Output:** one result file, **`Final_Result.xlsx`**, with the final modelling dataset, the feature dataset,
the data dictionary and all validation evidence, each on its own sheet

| Section | What is validated |
|---|---|
| A | Cleaned transaction dataset |
| B | Customer IDs and reference dates |
| C | Missing values, duplicate customers and invalid records in the modelling dataset |
| D | Feature and target recalculation (row by row) |
| E | No data leakage |
| F | Feature dataset, modelling dataset and data dictionary agree |
| G | Proof that the checks catch mistakes |

## 1. Setup and load all files

In [1]:
import os
import pandas as pd
import numpy as np

WORK_DIR = 'Working_Data'
FINAL_DIR = WORK_DIR      # datasets saved by notebook 02
pd.set_option('display.max_colwidth', 120)

raw = pd.read_csv('online_retail_II.csv', dtype={'Invoice': str, 'StockCode': str})
tx = pd.read_csv(f'{WORK_DIR}/cleaned_transactions.csv.gz', dtype={'Invoice': str, 'StockCode': str}, parse_dates=['InvoiceDate'])
cx = pd.read_csv(f'{WORK_DIR}/cancelled_transactions.csv.gz', dtype={'Invoice': str, 'StockCode': str}, parse_dates=['InvoiceDate'])
cleaning_log = pd.read_csv(f'{WORK_DIR}/cleaning_log.csv')
nb1_verification = pd.read_csv(f'{WORK_DIR}/transaction_verification.csv')
modelling = pd.read_csv(f'{FINAL_DIR}/Final_Modelling_Dataset.csv', parse_dates=['reference_date'])
features = pd.read_csv(f'{FINAL_DIR}/Feature_Dataset.csv', parse_dates=['reference_date'])
dictionary = pd.read_csv(f'{FINAL_DIR}/Feature_Data_Dictionary.csv')

for name, t in [('raw', raw), ('cleaned transactions', tx), ('cancelled transactions', cx),
                ('final modelling dataset', modelling), ('feature dataset', features), ('data dictionary', dictionary)]:
    print(f'{name:25s}: {t.shape[0]:>9,} rows x {t.shape[1]} columns')

raw                      : 1,067,371 rows x 8 columns
cleaned transactions     : 1,003,338 rows x 9 columns
cancelled transactions   :    17,914 rows x 9 columns
final modelling dataset  :    16,495 rows x 22 columns
feature dataset          :    16,495 rows x 21 columns
data dictionary          :        22 rows x 3 columns


In [2]:
# the expected design (from the project plan, not from notebook 02)
EXPECTED_REFERENCE_DATES = pd.to_datetime(['2010-12-01', '2011-03-01', '2011-06-01', '2011-09-01'])
OUT_OF_TIME_DATE = EXPECTED_REFERENCE_DATES.max()
OBS_MONTHS, TARGET_MONTHS, MIN_TENURE_DAYS, TENURE_CAP = 12, 3, 30, 180
TARGET = 'churn_next_3m'
ID_COLS = ['customer_id', 'reference_date', 'sample']
FEATURES = [c for c in modelling.columns if c not in ID_COLS + [TARGET]]
print('Features found:', len(FEATURES))

all_checks = []
def record(section, results):
    for name, (passed, detail) in results.items():
        all_checks.append({'Section': section, 'Check': name, 'Result': 'PASS' if passed else 'FAIL', 'Detail': str(detail)})
    t = pd.DataFrame([{'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in results.items()])
    print(f'{section}: {(t["Result"] == "PASS").sum()} out of {len(t)} passed')
    return t

Features found: 18


## A. Cleaned transaction dataset

In [3]:
tx['CustomerID'] = tx['CustomerID'].astype('Int64')
key = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Country', 'Revenue']
removed = cleaning_log.loc[cleaning_log['Step'] != 'TOTAL', 'Rows removed'].sum()
ids = tx['CustomerID'].dropna()

section_a = record('A. Cleaned transactions', {
    'Notebook 01 verification: all checks passed': ((nb1_verification['Result'] == 'PASS').all(),
                                                    f"{(nb1_verification['Result'] == 'PASS').sum()} of {len(nb1_verification)}"),
    'Raw rows = clean rows + removed rows (cleaning log)': (len(raw) == len(tx) + removed, f'{len(raw):,} = {len(tx):,} + {removed:,}'),
    'Cleaning log total matches the clean row count': (cleaning_log['Rows left'].iloc[-1] == len(tx), cleaning_log['Rows left'].iloc[-1]),
    'No missing values in key columns': (tx[key].isnull().sum().sum() == 0, tx[key].isnull().sum().sum()),
    'No duplicate rows': (tx.duplicated().sum() == 0, tx.duplicated().sum()),
    'Quantity > 0 and Price > 0': (((tx['Quantity'] > 0) & (tx['Price'] > 0)).all(), ''),
    'Invoice is 6 digits (no cancelled / adjustment invoices)': (tx['Invoice'].str.fullmatch(r'\d{6}').all(), ''),
    'CustomerID is a 5-digit number when present': (ids.between(10000, 99999).all(), f'{ids.min()} to {ids.max()}'),
    'Revenue = Quantity x Price': ((tx['Revenue'] - tx['Quantity'] * tx['Price']).abs().max() < 0.006, ''),
    'Dates between 01-Dec-2009 and 09-Dec-2011': (tx['InvoiceDate'].between('2009-12-01', '2011-12-10').all(),
                                                  f"{tx['InvoiceDate'].min().date()} to {tx['InvoiceDate'].max().date()}"),
    'Cancelled file: C invoices with negative quantity only': ((cx['Invoice'].str.startswith('C').all()
                                                               and (cx['Quantity'] < 0).all()), f'{len(cx)} lines'),
})
section_a

A. Cleaned transactions: 11 out of 11 passed


,Check,Result,Detail
0,Notebook 01 verification: all checks passed,PASS,22 of 22
1,Raw rows = clean rows + removed rows (cleaning log),PASS,"1,067,371 = 1,003,338 + 64,033"
2,Cleaning log total matches the clean row count,PASS,1003338
3,No missing values in key columns,PASS,0
4,No duplicate rows,PASS,0
5,Quantity > 0 and Price > 0,PASS,
6,Invoice is 6 digits (no cancelled / adjustment invoices),PASS,
7,CustomerID is a 5-digit number when present,PASS,12346 to 18287
8,Revenue = Quantity x Price,PASS,
9,Dates between 01-Dec-2009 and 09-Dec-2011,PASS,2009-12-01 to 2011-12-09


## Independent recalculation of the modelling dataset

Used by sections B, D and E. For each reference date, the code below is given **only the transactions before that date**
to build the customers and features, and **only the 3-month target window** to build the churn target.

In [4]:
known = tx.dropna(subset=['CustomerID']).copy()
known['CustomerID'] = known['CustomerID'].astype(int)
known['invoice_time'] = known.groupby('Invoice')['InvoiceDate'].transform('min')
known['invoice_day'] = known['invoice_time'].dt.normalize()
cancels = cx.dropna(subset=['CustomerID']).copy()
cancels['CustomerID'] = cancels['CustomerID'].astype(int)


def recalculate(ref):
    obs_start = ref - pd.DateOffset(months=OBS_MONTHS)
    past = known[known['invoice_time'] < ref]                     # nothing on or after the reference date
    first_day = past.groupby('CustomerID')['invoice_day'].min()
    obs = past[past['invoice_time'] >= obs_start]
    customers = sorted(c for c in obs['CustomerID'].unique() if (ref - first_day[c]).days >= MIN_TENURE_DAYS)
    obs = obs[obs['CustomerID'].isin(customers)]

    per = obs.groupby('CustomerID')
    r = pd.DataFrame(index=pd.Index(customers, name='customer_id'))
    r['recency_days'] = (ref - per['invoice_day'].max()).dt.days
    r['frequency_12m'] = per['Invoice'].nunique()
    for months, col in [(3, 'orders_last_3m'), (6, 'orders_last_6m')]:
        r[col] = obs[obs['invoice_time'] >= ref - pd.DateOffset(months=months)].groupby('CustomerID')['Invoice'].nunique()
    r['active_months_12m'] = per['invoice_time'].agg(lambda s: s.dt.to_period('M').nunique())
    span = (per['invoice_day'].max() - per['invoice_day'].min()).dt.days
    r['avg_days_between_orders_12m'] = np.where(r['frequency_12m'] > 1, (span / (r['frequency_12m'] - 1)).round(1), 365.0)
    r['is_one_time_buyer_12m'] = (r['frequency_12m'] == 1).astype(int)
    r['monetary_12m'] = per['Revenue'].sum().round(2)
    r['avg_order_value_12m'] = (r['monetary_12m'] / r['frequency_12m']).round(2)
    r['revenue_last_3m'] = obs[obs['invoice_time'] >= ref - pd.DateOffset(months=3)].groupby('CustomerID')['Revenue'].sum().round(2)
    r['units_12m'] = per['Quantity'].sum()
    r['revenue_trend_ratio'] = (r['revenue_last_3m'].fillna(0) / (r['monetary_12m'] / 4)).round(3)
    r['distinct_products_12m'] = per['StockCode'].nunique()
    r['tenure_days_capped'] = (ref - first_day.reindex(customers)).dt.days.clip(upper=TENURE_CAP)
    c_obs = cancels[(cancels['InvoiceDate'] >= obs_start) & (cancels['InvoiceDate'] < ref)]
    r['cancelled_orders_12m'] = c_obs.groupby('CustomerID')['Invoice'].nunique()
    r['orders_same_period_last_year'] = obs[obs['invoice_time'] < obs_start + pd.DateOffset(months=TARGET_MONTHS)].groupby('CustomerID')['Invoice'].nunique()
    r['is_uk'] = (past.sort_values('invoice_time').groupby('CustomerID')['Country'].last().reindex(customers) == 'United Kingdom').astype(int)
    r = r.fillna({'orders_last_3m': 0, 'orders_last_6m': 0, 'revenue_last_3m': 0, 'cancelled_orders_12m': 0,
                  'orders_same_period_last_year': 0})
    r['cancellation_rate_12m'] = (r['cancelled_orders_12m'] / (r['cancelled_orders_12m'] + r['frequency_12m'])).round(3)

    # target: only the 3 months after the reference date
    target_window = known[(known['invoice_time'] >= ref) & (known['invoice_time'] < ref + pd.DateOffset(months=TARGET_MONTHS))]
    r[TARGET] = (~r.index.isin(target_window['CustomerID'])).astype(int)

    r = r.reset_index()
    r['reference_date'] = ref
    return r, past['invoice_time'].max(), target_window['invoice_time'].min(), target_window['invoice_time'].max()

expected_parts, data_used = [], []
for ref in EXPECTED_REFERENCE_DATES:
    part, last_feature_time, first_target_time, last_target_time = recalculate(ref)
    expected_parts.append(part)
    data_used.append({'reference_date': ref.date(), 'customers': len(part),
                      'latest data used for features': last_feature_time,
                      'first data used for target': first_target_time, 'last data used for target': last_target_time})
expected = pd.concat(expected_parts, ignore_index=True)
data_used = pd.DataFrame(data_used)
print('Independently recalculated rows:', len(expected))
data_used

Independently recalculated rows: 16495


,reference_date,customers,latest data used for features,first data used for target,last data used for target
0,2010-12-01,3921,2010-11-30 19:35:00,2010-12-01 08:26:00,2011-02-28 17:01:00
1,2011-03-01,4132,2011-02-28 17:01:00,2011-03-01 08:30:00,2011-05-31 15:41:00
2,2011-06-01,4214,2011-05-31 15:41:00,2011-06-01 07:37:00,2011-08-31 17:16:00
3,2011-09-01,4228,2011-08-31 17:16:00,2011-09-01 08:25:00,2011-11-30 17:37:00


## Validation function

All checks for sections B to F are in one function, so section G can run exactly the same checks on data with mistakes in it.

In [5]:
tx_customers = set(known['CustomerID'])
first_partial_month = tx['InvoiceDate'].max().to_period('M').to_timestamp()
TOLERANCE = {'monetary_12m': 0.01, 'avg_order_value_12m': 0.01, 'revenue_last_3m': 0.01,
             'revenue_trend_ratio': 0.001, 'cancellation_rate_12m': 0.001, 'avg_days_between_orders_12m': 0.05}


def validate(md_, fd_, dictionary_=dictionary):
    out = {}
    keys = ['customer_id', 'reference_date']
    feats = [c for c in md_.columns if c not in ID_COLS + [TARGET]]
    m = md_.merge(expected, on=keys, how='outer', suffixes=('', '_expected'), indicator=True)
    both = m[m['_merge'] == 'both']

    # ---------------- B. customer IDs and reference dates
    b = {}
    cid = pd.to_numeric(md_['customer_id'], errors='coerce')
    b['customer_id is never empty'] = (md_['customer_id'].notnull().all(), f"{md_['customer_id'].isnull().sum()} empty")
    b['customer_id is a whole 5-digit number'] = ((cid % 1 == 0).all() and cid.between(10000, 99999).all(),
                                                  f'{cid.min():.0f} to {cid.max():.0f}')
    unknown = set(cid.dropna().astype(int)) - tx_customers
    b['Every customer_id exists in the cleaned transactions'] = (len(unknown) == 0, f'{len(unknown)} unknown')
    refs = set(md_['reference_date'].dropna())
    b['Reference dates are exactly the 4 planned dates'] = (refs == set(EXPECTED_REFERENCE_DATES),
                                                            ', '.join(sorted(str(d.date()) for d in refs)))
    b['Reference dates are the 1st of a month'] = (all(d.day == 1 for d in refs), '')
    b['Every target window ends before the partial month (Dec-2011)'] = (
        all(d + pd.DateOffset(months=TARGET_MONTHS) <= first_partial_month for d in refs), f'partial month starts {first_partial_month.date()}')
    b['Every observation window starts on or after the first data date'] = (
        all(d - pd.DateOffset(months=OBS_MONTHS) >= tx['InvoiceDate'].min().normalize() for d in refs), '')
    sample_ok = np.where(md_['reference_date'] == OUT_OF_TIME_DATE, 'out_of_time_test', 'development')
    b['sample label matches the reference date'] = ((md_['sample'] == sample_ok).all(), '')
    b['Rows = eligible customers (no missing, no extra customers)'] = (
        (m['_merge'] == 'both').all(), f"{(m['_merge'] == 'left_only').sum()} extra, {(m['_merge'] == 'right_only').sum()} missing")
    counts = md_.groupby('reference_date').size().reindex(EXPECTED_REFERENCE_DATES).fillna(0)
    b['Rows per reference date = independent eligibility recount'] = (
        (counts.values == expected.groupby('reference_date').size().reindex(EXPECTED_REFERENCE_DATES).values).all(),
        ', '.join(str(int(x)) for x in counts.values))
    out['B. Customer IDs and reference dates'] = b

    # ---------------- C. missing values, duplicate customers, invalid records
    c = {}
    x = md_
    c['No missing values in any column'] = (x.isnull().sum().sum() == 0, f'{x.isnull().sum().sum()} missing')
    c['No duplicate customer at the same reference date'] = (not x.duplicated(keys).any(), f'{x.duplicated(keys).sum()} duplicates')
    c['No fully duplicated rows'] = (not x.duplicated().any(), f'{x.duplicated().sum()} duplicates')
    def rule(name, ok):
        ok = ok.fillna(False)
        c[name] = (ok.all(), f'{(~ok).sum()} invalid rows')
    rule('recency_days between 1 and 365', x['recency_days'].between(1, 365))
    rule('frequency_12m >= 1', x['frequency_12m'] >= 1)
    rule('orders_last_3m <= orders_last_6m <= frequency_12m', (x['orders_last_3m'] <= x['orders_last_6m']) & (x['orders_last_6m'] <= x['frequency_12m']))
    rule('active_months_12m between 1 and 12 and <= frequency', x['active_months_12m'].between(1, 12) & (x['active_months_12m'] <= x['frequency_12m']))
    rule('avg_days_between_orders_12m between 0 and 365', x['avg_days_between_orders_12m'].between(0, 365))
    rule('is_one_time_buyer_12m = 1 only when frequency is 1', x['is_one_time_buyer_12m'] == (x['frequency_12m'] == 1).astype(int))
    rule('monetary_12m > 0', x['monetary_12m'] > 0)
    rule('avg_order_value_12m = monetary / frequency', (x['avg_order_value_12m'] - x['monetary_12m'] / x['frequency_12m']).abs() < 0.01)
    rule('revenue_last_3m between 0 and monetary_12m', (x['revenue_last_3m'] >= 0) & (x['revenue_last_3m'] <= x['monetary_12m'] + 0.01))
    rule('units_12m >= 1', x['units_12m'] >= 1)
    rule('revenue_trend_ratio between 0 and 4', x['revenue_trend_ratio'].between(0, 4))
    rule('distinct_products_12m >= 1', x['distinct_products_12m'] >= 1)
    rule(f'tenure_days_capped between {MIN_TENURE_DAYS} and {TENURE_CAP}', x['tenure_days_capped'].between(MIN_TENURE_DAYS, TENURE_CAP))
    rule('cancelled_orders_12m >= 0', x['cancelled_orders_12m'] >= 0)
    rule('cancellation_rate_12m between 0 and 1', x['cancellation_rate_12m'].between(0, 1))
    rule('orders_same_period_last_year <= frequency_12m', x['orders_same_period_last_year'] <= x['frequency_12m'])
    rule('is_uk is 0 or 1', x['is_uk'].isin([0, 1]))
    rule('churn_next_3m is 0 or 1', x[TARGET].isin([0, 1]))
    out['C. Missing values, duplicates, invalid records'] = c

    # ---------------- D. recalculation
    d = {}
    for col in FEATURES + [TARGET]:
        if col not in md_.columns:
            d[f'{col} matches the recalculation'] = (False, 'column missing')
            continue
        diff = (pd.to_numeric(both[col], errors='coerce') - both[f'{col}_expected']).abs()
        bad = (diff > TOLERANCE.get(col, 0)) | diff.isnull()
        d[f'{col} matches the recalculation'] = (len(both) > 0 and not bad.any(), f'{bad.sum()} of {len(both)} rows differ')
    out['D. Feature and target recalculation'] = d

    # ---------------- E. leakage
    e = {}
    e['Recalculation used only data before each reference date'] = (
        all(pd.Timestamp(r['latest data used for features']) < pd.Timestamp(r['reference_date']) for _, r in data_used.iterrows()), '')
    e['Every feature equals its no-future-data recalculation'] = (all(v[0] for v in d.values()), 'see section D')
    e['Target uses only the 3 months after the reference date'] = (
        all(pd.Timestamp(r['first data used for target']) >= pd.Timestamp(r['reference_date']) and
            pd.Timestamp(r['last data used for target']) < pd.Timestamp(r['reference_date']) + pd.DateOffset(months=TARGET_MONTHS)
            for _, r in data_used.iterrows()), '')
    dev_end = max(d_ + pd.DateOffset(months=TARGET_MONTHS) for d_ in refs if d_ != OUT_OF_TIME_DATE) if len(refs) > 1 else None
    e['Development target windows end before the out-of-time reference date'] = (dev_end is not None and dev_end <= OUT_OF_TIME_DATE, str(dev_end))
    corr = md_[feats].apply(pd.to_numeric, errors='coerce').corrwith(pd.to_numeric(md_[TARGET], errors='coerce')).abs()
    e['No feature is almost the target (|corr| < 0.9)'] = ((corr < 0.9).all(), f'max {corr.max():.2f} ({corr.idxmax()})')
    e['Feature dataset has no target column'] = (TARGET not in fd_.columns, '')
    out['E. No data leakage'] = e

    # ---------------- F. consistency
    f = {}
    f['Feature dataset has the same rows as the modelling dataset'] = (
        len(fd_) == len(md_) and fd_[keys].reset_index(drop=True).equals(md_[keys].reset_index(drop=True)), f'{len(fd_)} vs {len(md_)}')
    common = [col for col in FEATURES if col in fd_.columns and col in md_.columns]
    f['Feature values are identical in both datasets'] = (
        len(fd_) == len(md_) and len(common) == len(FEATURES)
        and fd_[common].reset_index(drop=True).equals(md_[common].reset_index(drop=True)), '')
    f['Data dictionary documents every column'] = (set(dictionary_['column']) == set(md_.columns),
                                                   f"{len(set(md_.columns) - set(dictionary_['column']))} undocumented")
    f['Columns are in data dictionary order'] = (list(dictionary_['column']) == list(md_.columns), '')
    out['F. Dataset consistency'] = f
    return out

results = validate(modelling, features)
section_tables = {name: record(name, checks) for name, checks in results.items()}

B. Customer IDs and reference dates: 10 out of 10 passed
C. Missing values, duplicates, invalid records: 21 out of 21 passed
D. Feature and target recalculation: 19 out of 19 passed
E. No data leakage: 6 out of 6 passed
F. Dataset consistency: 4 out of 4 passed


## B. Customer IDs and reference dates

In [6]:
section_tables['B. Customer IDs and reference dates']

,Check,Result,Detail
0,customer_id is never empty,PASS,0 empty
1,customer_id is a whole 5-digit number,PASS,12346 to 18287
2,Every customer_id exists in the cleaned transactions,PASS,0 unknown
3,Reference dates are exactly the 4 planned dates,PASS,"2010-12-01, 2011-03-01, 2011-06-01, 2011-09-01"
4,Reference dates are the 1st of a month,PASS,
5,Every target window ends before the partial month (Dec-2011),PASS,partial month starts 2011-12-01
6,Every observation window starts on or after the first data date,PASS,
7,sample label matches the reference date,PASS,
8,"Rows = eligible customers (no missing, no extra customers)",PASS,"0 extra, 0 missing"
9,Rows per reference date = independent eligibility recount,PASS,"3921, 4132, 4214, 4228"


## C. Missing values, duplicates, invalid records

In [7]:
section_tables['C. Missing values, duplicates, invalid records']

,Check,Result,Detail
0,No missing values in any column,PASS,0 missing
1,No duplicate customer at the same reference date,PASS,0 duplicates
2,No fully duplicated rows,PASS,0 duplicates
3,recency_days between 1 and 365,PASS,0 invalid rows
4,frequency_12m >= 1,PASS,0 invalid rows
5,orders_last_3m <= orders_last_6m <= frequency_12m,PASS,0 invalid rows
6,active_months_12m between 1 and 12 and <= frequency,PASS,0 invalid rows
7,avg_days_between_orders_12m between 0 and 365,PASS,0 invalid rows
8,is_one_time_buyer_12m = 1 only when frequency is 1,PASS,0 invalid rows
9,monetary_12m > 0,PASS,0 invalid rows


## D. Feature and target recalculation

In [8]:
section_tables['D. Feature and target recalculation']

,Check,Result,Detail
0,recency_days matches the recalculation,PASS,0 of 16495 rows differ
1,frequency_12m matches the recalculation,PASS,0 of 16495 rows differ
2,orders_last_3m matches the recalculation,PASS,0 of 16495 rows differ
3,orders_last_6m matches the recalculation,PASS,0 of 16495 rows differ
4,active_months_12m matches the recalculation,PASS,0 of 16495 rows differ
5,avg_days_between_orders_12m matches the recalculation,PASS,0 of 16495 rows differ
6,is_one_time_buyer_12m matches the recalculation,PASS,0 of 16495 rows differ
7,monetary_12m matches the recalculation,PASS,0 of 16495 rows differ
8,avg_order_value_12m matches the recalculation,PASS,0 of 16495 rows differ
9,revenue_last_3m matches the recalculation,PASS,0 of 16495 rows differ


## E. No data leakage

In [9]:
section_tables['E. No data leakage']

,Check,Result,Detail
0,Recalculation used only data before each reference date,PASS,
1,Every feature equals its no-future-data recalculation,PASS,see section D
2,Target uses only the 3 months after the reference date,PASS,
3,Development target windows end before the out-of-time reference date,PASS,2011-09-01 00:00:00
4,No feature is almost the target (|corr| < 0.9),PASS,max 0.47 (active_months_12m)
5,Feature dataset has no target column,PASS,


## F. Dataset consistency

In [10]:
section_tables['F. Dataset consistency']

,Check,Result,Detail
0,Feature dataset has the same rows as the modelling dataset,PASS,16495 vs 16495
1,Feature values are identical in both datasets,PASS,
2,Data dictionary documents every column,PASS,0 undocumented
3,Columns are in data dictionary order,PASS,


### Reports for missing values, duplicate customers and invalid records

In [11]:
missing_report = pd.DataFrame({'missing_in_modelling_dataset': modelling.isnull().sum(),
                               'missing_in_feature_dataset': features.isnull().sum().reindex(modelling.columns)}).fillna('n/a (target)')
missing_report

,missing_in_modelling_dataset,missing_in_feature_dataset
customer_id,0,0.0
reference_date,0,0.0
sample,0,0.0
recency_days,0,0.0
frequency_12m,0,0.0
orders_last_3m,0,0.0
orders_last_6m,0,0.0
active_months_12m,0,0.0
avg_days_between_orders_12m,0,0.0
is_one_time_buyer_12m,0,0.0


In [12]:
appearances = modelling.groupby('customer_id')['reference_date'].nunique().value_counts().sort_index()
duplicate_report = pd.DataFrame({
    'Check': ['Duplicate (customer_id, reference_date) rows', 'Fully duplicated rows'] +
             [f'Customers appearing at {k} reference date(s) (expected, one row per date)' for k in appearances.index],
    'Count': [int(modelling.duplicated(['customer_id', 'reference_date']).sum()), int(modelling.duplicated().sum())] +
             [int(v) for v in appearances.values]})
duplicate_report

,Check,Count
0,"Duplicate (customer_id, reference_date) rows",0
1,Fully duplicated rows,0
2,"Customers appearing at 1 reference date(s) (expected, one row per date)",557
3,"Customers appearing at 2 reference date(s) (expected, one row per date)",750
4,"Customers appearing at 3 reference date(s) (expected, one row per date)",850
5,"Customers appearing at 4 reference date(s) (expected, one row per date)",2972


In [13]:
invalid_report = pd.DataFrame([{'Rule': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Invalid rows': v[1]}
                               for k, v in results['C. Missing values, duplicates, invalid records'].items()])
invalid_report

,Rule,Result,Invalid rows
0,No missing values in any column,PASS,0 missing
1,No duplicate customer at the same reference date,PASS,0 duplicates
2,No fully duplicated rows,PASS,0 duplicates
3,recency_days between 1 and 365,PASS,0 invalid rows
4,frequency_12m >= 1,PASS,0 invalid rows
5,orders_last_3m <= orders_last_6m <= frequency_12m,PASS,0 invalid rows
6,active_months_12m between 1 and 12 and <= frequency,PASS,0 invalid rows
7,avg_days_between_orders_12m between 0 and 365,PASS,0 invalid rows
8,is_one_time_buyer_12m = 1 only when frequency is 1,PASS,0 invalid rows
9,monetary_12m > 0,PASS,0 invalid rows


In [14]:
recalc_report = pd.DataFrame([{'Column': k.replace(' matches the recalculation', ''), 'Result': 'PASS' if v[0] else 'FAIL',
                               'Rows compared / differing': v[1],
                               'Tolerance': TOLERANCE.get(k.replace(' matches the recalculation', ''), 0)}
                              for k, v in results['D. Feature and target recalculation'].items()])
eligibility_recount = pd.DataFrame({'reference_date': EXPECTED_REFERENCE_DATES.date,
                                    'rows_in_modelling_dataset': modelling.groupby('reference_date').size().reindex(EXPECTED_REFERENCE_DATES).values,
                                    'independent_recount': expected.groupby('reference_date').size().reindex(EXPECTED_REFERENCE_DATES).values})
eligibility_recount['match'] = eligibility_recount['rows_in_modelling_dataset'] == eligibility_recount['independent_recount']
eligibility_recount

,reference_date,rows_in_modelling_dataset,independent_recount,match
0,2010-12-01,3921,3921,True
1,2011-03-01,4132,4132,True
2,2011-06-01,4214,4214,True
3,2011-09-01,4228,4228,True


## G. Proof that the checks catch mistakes

A check that always says PASS proves nothing. Here I put one mistake into a copy of the datasets and run the same `validate` function.
Every mistake must make at least one check FAIL.

In [15]:
mistake_tests = []
def try_mistake(name, md_=None, fd_=None):
    res = validate(modelling if md_ is None else md_, features if fd_ is None else fd_)
    failed = [k for sec in res.values() for k, v in sec.items() if not v[0]]
    mistake_tests.append({'Mistake added': name, 'Checks failed': len(failed), 'Caught': len(failed) > 0,
                          'Failed checks': '; '.join(failed)})

def broken(fn):
    m = modelling.copy(); fn(m); return m

try_mistake('Same customer twice at one reference date', pd.concat([modelling, modelling.iloc[[0]]], ignore_index=True))
try_mistake('Empty value in a feature', broken(lambda m: m.__setitem__('monetary_12m', m['monetary_12m'].mask(m.index == 5))))
try_mistake('Customer ID that does not exist (99999)', broken(lambda m: m.__setitem__('customer_id', m['customer_id'].mask(m.index == 5, 99999))))
try_mistake('Wrong reference date (15-Sep-2011)', broken(lambda m: m.__setitem__('reference_date', m['reference_date'].mask(m.index == 5, pd.Timestamp('2011-09-15')))))
try_mistake('Churn value of 2', broken(lambda m: m.__setitem__(TARGET, m[TARGET].mask(m.index == 5, 2))))
try_mistake('Negative spend', broken(lambda m: m.__setitem__('monetary_12m', m['monetary_12m'].mask(m.index == 5, -10.0))))
try_mistake('Recency longer than 12 months', broken(lambda m: m.__setitem__('recency_days', m['recency_days'].mask(m.index == 5, 400))))
try_mistake('Future order counted in frequency (leakage)',
            broken(lambda m: m.__setitem__('frequency_12m', m['frequency_12m'] + (1 - m[TARGET]))))
new_customer = sorted(tx_customers - set(modelling.loc[modelling['reference_date'] == OUT_OF_TIME_DATE, 'customer_id']))[0]
extra = modelling[modelling['reference_date'] == OUT_OF_TIME_DATE].iloc[[0]].assign(customer_id=new_customer)
try_mistake('Customer who was not eligible at the reference date', pd.concat([modelling, extra], ignore_index=True))
try_mistake('Feature dataset out of sync with the modelling dataset',
            fd_=features.assign(recency_days=features['recency_days'].mask(features.index == 5, 1)))
try_mistake('Target column left in the feature dataset', fd_=features.assign(**{TARGET: modelling[TARGET]}))
try_mistake('Target copied into a feature', broken(lambda m: m.__setitem__('is_uk', m[TARGET])))

mistake_tests = pd.DataFrame(mistake_tests)
print('Mistakes caught:', mistake_tests['Caught'].sum(), 'out of', len(mistake_tests))
mistake_tests

Mistakes caught: 12 out of 12


,Mistake added,Checks failed,Caught,Failed checks
0,Same customer twice at one reference date,5,True,Rows per reference date = independent eligibility recount; No duplicate customer at the same reference date; No full...
1,Empty value in a feature,7,True,No missing values in any column; monetary_12m > 0; avg_order_value_12m = monetary / frequency; revenue_last_3m betwe...
2,Customer ID that does not exist (99999),3,True,"Every customer_id exists in the cleaned transactions; Rows = eligible customers (no missing, no extra customers); Fe..."
3,Wrong reference date (15-Sep-2011),7,True,Reference dates are exactly the 4 planned dates; Reference dates are the 1st of a month; Every target window ends be...
4,Churn value of 2,3,True,churn_next_3m is 0 or 1; churn_next_3m matches the recalculation; Every feature equals its no-future-data recalculation
5,Negative spend,6,True,monetary_12m > 0; avg_order_value_12m = monetary / frequency; revenue_last_3m between 0 and monetary_12m; monetary_1...
6,Recency longer than 12 months,4,True,recency_days between 1 and 365; recency_days matches the recalculation; Every feature equals its no-future-data reca...
7,Future order counted in frequency (leakage),5,True,is_one_time_buyer_12m = 1 only when frequency is 1; avg_order_value_12m = monetary / frequency; frequency_12m matche...
8,Customer who was not eligible at the reference date,4,True,"Rows = eligible customers (no missing, no extra customers); Rows per reference date = independent eligibility recoun..."
9,Feature dataset out of sync with the modelling dataset,1,True,Feature values are identical in both datasets


## Final result file: datasets and validation evidence

In [16]:
checks_table = pd.DataFrame(all_checks)
summary = checks_table.groupby('Section', sort=False)['Result'].agg(
    checks='count', passed=lambda s: (s == 'PASS').sum()).reset_index()
summary['result'] = np.where(summary['checks'] == summary['passed'], 'PASS', 'FAIL')
summary = pd.concat([summary, pd.DataFrame([
    {'Section': 'G. Mistakes caught by the checks', 'checks': len(mistake_tests),
     'passed': int(mistake_tests['Caught'].sum()),
     'result': 'PASS' if mistake_tests['Caught'].all() else 'FAIL'},
    {'Section': 'TOTAL', 'checks': summary['checks'].sum() + len(mistake_tests),
     'passed': summary['passed'].sum() + int(mistake_tests['Caught'].sum()), 'result': ''}])], ignore_index=True)
summary.loc[summary['Section'] == 'TOTAL', 'result'] = 'PASS' if (summary['result'][:-1] == 'PASS').all() else 'FAIL'
summary

,Section,checks,passed,result
0,A. Cleaned transactions,11,11,PASS
1,B. Customer IDs and reference dates,10,10,PASS
2,"C. Missing values, duplicates, invalid records",21,21,PASS
3,D. Feature and target recalculation,19,19,PASS
4,E. No data leakage,6,6,PASS
5,F. Dataset consistency,4,4,PASS
6,G. Mistakes caught by the checks,12,12,PASS
7,TOTAL,83,83,PASS


In [17]:
RESULT_FILE = 'Final_Result.xlsx'
churn = modelling.groupby('reference_date')[TARGET].mean().mul(100).round(1)
readme = pd.DataFrame([
    ('Project', 'Online Retail II - Customer Churn Prediction'),
    ('Role', 'Data Engineering & Cleaning'),
    ('Created by', 'Notebooks 01_Data_Verification -> 02_Modelling_and_Feature_Dataset -> 03_Data_Validation'),
    ('Churn definition', 'churn_next_3m = 1 if the customer places no order in the 3 months after the reference date'),
    ('Reference dates', ', '.join(f"{d:%d-%b-%Y}" for d in EXPECTED_REFERENCE_DATES) + ' (last one = out-of-time test)'),
    ('Rows / customers', f"{len(modelling):,} rows, {modelling['customer_id'].nunique():,} different customers"),
    ('Features', f'{len(FEATURES)} churn features'),
    ('Churn rate by reference date', ', '.join(f"{d:%b-%Y}: {v}%" for d, v in churn.items())),
    ('Validation result', f"{(checks_table['Result'] == 'PASS').sum()} of {len(checks_table)} checks passed, "
                          f"{int(mistake_tests['Caught'].sum())} of {len(mistake_tests)} deliberate mistakes caught"),
    ('', ''),
    ('SHEET', 'CONTENT'),
    ('Final_Modelling_Dataset', 'DELIVERABLE 1 - customer ID, reference date, sample, 18 features and the churn target'),
    ('Feature_Dataset', 'DELIVERABLE 2 - the same rows and features, without the target'),
    ('Data_Dictionary', 'Meaning of every column'),
    ('Validation_Summary', 'DELIVERABLE 3 - data validation evidence: result of every section'),
    ('All_Checks', 'Every validation check with its result and detail'),
    ('Transaction_Checks', 'Checks of the cleaned transaction dataset'),
    ('Missing_Values', 'Missing values per column'),
    ('Duplicate_Customers', 'Duplicate customers and rows'),
    ('Invalid_Records', 'Number of invalid rows for each rule'),
    ('Feature_Recalculation', 'Row-by-row comparison with an independent recalculation'),
    ('Eligibility_Recount', 'Customers per reference date vs an independent recount'),
    ('Data_Windows', 'Dates of the data used for features and target (no leakage)'),
    ('Mistake_Tests', 'Deliberate mistakes and the checks that caught them'),
    ('Cleaning_Log', 'Rows removed in cleaning and why'),
], columns=['Item', 'Value'])

sheets = {
    'README': readme,
    'Final_Modelling_Dataset': modelling.assign(reference_date=modelling['reference_date'].dt.date),
    'Feature_Dataset': features.assign(reference_date=features['reference_date'].dt.date),
    'Data_Dictionary': dictionary,
    'Validation_Summary': summary,
    'All_Checks': checks_table,
    'Transaction_Checks': section_a,
    'Missing_Values': missing_report.reset_index(names='column'),
    'Duplicate_Customers': duplicate_report,
    'Invalid_Records': invalid_report,
    'Feature_Recalculation': recalc_report,
    'Eligibility_Recount': eligibility_recount,
    'Data_Windows': data_used,
    'Mistake_Tests': mistake_tests,
    'Cleaning_Log': cleaning_log,
}
with pd.ExcelWriter(RESULT_FILE) as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)

saved = pd.read_excel(RESULT_FILE, sheet_name=None)
for name, table in sheets.items():
    assert saved[name].shape == table.shape, f'sheet {name} is wrong'
check_back = saved['Final_Modelling_Dataset'].assign(reference_date=pd.to_datetime(saved['Final_Modelling_Dataset']['reference_date']))
assert (check_back[FEATURES + [TARGET, 'customer_id']].round(6) == modelling[FEATURES + [TARGET, 'customer_id']].round(6)).all().all()
print('Saved', RESULT_FILE, 'with', len(saved), 'sheets:', ', '.join(saved))
print('Final_Modelling_Dataset sheet:', saved['Final_Modelling_Dataset'].shape, '| Feature_Dataset sheet:', saved['Feature_Dataset'].shape)

Saved Final_Result.xlsx with 15 sheets: README, Final_Modelling_Dataset, Feature_Dataset, Data_Dictionary, Validation_Summary, All_Checks, Transaction_Checks, Missing_Values, Duplicate_Customers, Invalid_Records, Feature_Recalculation, Eligibility_Recount, Data_Windows, Mistake_Tests, Cleaning_Log
Final_Modelling_Dataset sheet: (16495, 22) | Feature_Dataset sheet: (16495, 21)


In [18]:
assert (checks_table['Result'] == 'PASS').all(), 'Some validation checks failed'
assert mistake_tests['Caught'].all(), 'A mistake was not caught'
print(f"VALIDATION COMPLETE: {(checks_table['Result'] == 'PASS').sum()} of {len(checks_table)} checks passed, "
      f"{int(mistake_tests['Caught'].sum())} of {len(mistake_tests)} mistakes caught.")
print('The final modelling dataset and feature dataset are verified and ready for churn modelling.')

VALIDATION COMPLETE: 71 of 71 checks passed, 12 of 12 mistakes caught.
The final modelling dataset and feature dataset are verified and ready for churn modelling.


## Summary: data validation evidence

| Section | Checks | Result |
|---|---|---|
| A. Cleaned transaction dataset | 11 | all PASS |
| B. Customer IDs and reference dates | 10 | all PASS |
| C. Missing values, duplicate customers, invalid records | 21 | all PASS |
| D. Feature and target recalculation (every row of all 18 features + target) | 19 | all PASS |
| E. No data leakage | 6 | all PASS |
| F. Dataset consistency (modelling dataset, feature dataset, data dictionary) | 4 | all PASS |
| G. Deliberate mistakes caught | 12 | all caught |
| **Total** | **83** | **83 PASS** |

- 0 missing values, 0 duplicate customers at a reference date, 0 invalid records in 16,495 rows
- Every customer ID exists in the cleaned transactions; every reference date is one of the 4 planned dates
- An independent recalculation that only sees data before each reference date gives exactly the same 16,495 rows, 18 features and churn target

**Final result file:** `Final_Result.xlsx` (15 sheets): README, Final_Modelling_Dataset, Feature_Dataset, Data_Dictionary
and 11 sheets of validation evidence

In [19]:
facts = [('Checks A-F', 71, len(checks_table)), ('Checks passed', 71, int((checks_table['Result'] == 'PASS').sum())),
         ('Section A', 11, len(section_a)), ('Section B', 10, len(results['B. Customer IDs and reference dates'])),
         ('Section C', 21, len(results['C. Missing values, duplicates, invalid records'])),
         ('Section D', 19, len(results['D. Feature and target recalculation'])), ('Section E', 6, len(results['E. No data leakage'])),
         ('Section F', 4, len(results['F. Dataset consistency'])), ('Mistakes caught', 12, int(mistake_tests['Caught'].sum())),
         ('Rows validated', 16495, len(modelling)), ('Result file sheets', 15, len(saved))]
facts = pd.DataFrame(facts, columns=['Item', 'Reported', 'Notebook output'])
facts['Match'] = facts['Reported'].astype(str) == facts['Notebook output'].astype(str)
print('Summary numbers matching the outputs:', facts['Match'].sum(), 'out of', len(facts))
assert facts['Match'].all()
facts

Summary numbers matching the outputs: 11 out of 11


,Item,Reported,Notebook output,Match
0,Checks A-F,71,71,True
1,Checks passed,71,71,True
2,Section A,11,11,True
3,Section B,10,10,True
4,Section C,21,21,True
5,Section D,19,19,True
6,Section E,6,6,True
7,Section F,4,4,True
8,Mistakes caught,12,12,True
9,Rows validated,16495,16495,True
